In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

#Desactivar los warnings
import warnings
warnings.filterwarnings("ignore")

# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'

# ── Variables de entorno ──────────────────────────────────────────────
load_dotenv(RAIZ / '.env')

print('✅ Entorno listo')
print(f'   Raíz del proyecto: {RAIZ}')

print("✅ Librerías importadas correctamente")

✅ Entorno listo
   Raíz del proyecto: c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Mis Casos(Importante)\PROYECTO 5 - FORECASTING RETAIL\02_FORECASTING
✅ Librerías importadas correctamente


## EMPAQUETAR TODO EL PROCESO EN FUNCIONES

### CALIDAD DE DATOS

In [2]:
#Variables madre segun el Excel de procesos: solo estas entran al proceso
VAR_MADRE = ['store_id', 'item_id', 'event_name_1', 'event_type_1', 'month', 'sell_price', 'ventas', 'weekday']

def calidad_datos(x):
    temp = x[VAR_MADRE].copy()
    #modificar tipos: solo month, porque month_2 es variable final y el OHE se entreno con month en texto
    #(year y wday no llegan a ninguna variable final, convertirlas no sirve para nada)
    temp['month'] = temp['month'].astype('str')
    #imputar nulos de eventos, event_type_1_Sin_evento es variable final
    temp.fillna({'event_name_1': 'Sin_evento', 'event_type_1': 'Sin_evento'}, inplace=True)
    #eliminar registros sin precio (pre lanzamiento, como los 71 de FOODS_3_288), sin reset_index para no perder date
    temp = temp.dropna(subset=['sell_price'])
    return temp

### CREACION DE VARIABLES

In [3]:
def crear_variables(x):
    #demanda intermitente
    def rotura_stock(ventas, n):
        a = (ventas==0).astype(int)
        b = a.rolling(n).sum()
        return(b==n).astype(int)

    #Por si acaso se ordena el dataframe antes de agrupar
    x = x.sort_values(['store_id', 'item_id', 'date'])
    nivelPxT = x.groupby(['store_id', 'item_id'])['ventas']

    for n in (3, 7, 15):
        x[f'rotura_stock_{n}'] = nivelPxT.transform(rotura_stock, n=n)

    # ── Lags ─────────────────────────────────────────────────────────────
    # 'columna_original': [lags en dias] -> para añadir/quitar lags solo se toca esto
    CONFIG_LAGS = {
        'ventas':          list(range(1, 16)),
        'sell_price':      list(range(1, 8)),
        'rotura_stock_3':  [1],
        'rotura_stock_7':  [1],
        'rotura_stock_15': [1],
    }

    #Creacion de lags
    def crear_lags(df, config, cols_serie=('store_id', 'item_id'), col_fecha='date'):
        """Lags dentro de cada serie producto-tienda. Devuelve un df de lags (sin unir)."""
        df = df.sort_values(list(cols_serie) + [col_fecha])
        grupos = df.groupby(list(cols_serie))
        resultado = pd.DataFrame(index=df.index)

        for columna in config:
            for lag in config[columna]:
                resultado[f'{columna}_lag{lag}'] = grupos[columna].shift(lag)

        return resultado

    df_lags = crear_lags(x, CONFIG_LAGS)

    #Ventanas moviles
    #Estadisticos min y max y media movil ponderada
    def estadistico_movil_serie(ventas, n, estadistico):
        return ventas.shift(1).rolling(n).agg(estadistico)

    def media_movil_ponderada_serie(ventas, n):
        return ventas.shift(1).ewm(span=n, min_periods=n).mean()

    for n in range(2, 16):
        x['ventasmin' + str(n)] = nivelPxT.transform(estadistico_movil_serie, n=n, estadistico='min')
        x['ventasmax' + str(n)] = nivelPxT.transform(estadistico_movil_serie, n=n, estadistico='max')
        x['ventasmmp' + str(n)] = nivelPxT.transform(media_movil_ponderada_serie, n=n)

    #unir todos los lags con el dataframe original
    x = pd.concat([x, df_lags], axis=1)
    #Eliminar las FILAS que quedaron con nulos por los lags (los primeros 15 dias de cada serie)
    #con subset para no depender de otras columnas con nulos como event_name_2
    x = x.dropna(subset=df_lags.columns)
    #Variables que ya no se van a necesitar (las originales)
    x = x.drop(columns=['sell_price','rotura_stock_3','rotura_stock_7','rotura_stock_15'])
    # Se crea una sola variable para el producto debido a que se haran 20 modelos, una por cada
    # producto-tienda, se eliminan entonces las columnas 'item_id' y 'store_id'
    # mismo orden que en la modelizacion (tienda_producto) para que coincida con lista_modelos
    x['producto'] = x['store_id'] + '_' + x['item_id']
    x = x.drop(columns=['item_id', 'store_id'])
    return x

### TRANSFORMACION DE VARIABLES

In [4]:
import joblib

# Las madres de las dummies finales (Excel de procesos)
VAR_OHE = ['event_name_1', 'event_type_1', 'month', 'weekday']
RUTA_OHE = MODELOS / 'ohe_retail.pkl'


def aplicar_ohe(x, ohe):
    # Cambia las categoricas por sus dummies; index=x.index para que cada dummy caiga en su fila
    dummies = pd.DataFrame(ohe.transform(x[VAR_OHE]), columns=ohe.get_feature_names_out(), index=x.index)
    return pd.concat([x.drop(columns=VAR_OHE), dummies], axis=1)


def transformar_variables_entrenamiento(x):
    # Entrenar: el encoder aprende las categorias de TODO el historico y se guarda
    ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore').fit(x[VAR_OHE])
    joblib.dump(ohe, RUTA_OHE)
    return aplicar_ohe(x, ohe)


def transformar_variables_ejecucion(x):
    # Ejecutar: se usa el encoder guardado y no uno nuevo, para que salgan las mismas columnas del entrenamiento
    ohe = joblib.load(RUTA_OHE)
    return aplicar_ohe(x, ohe)

### PRESELECCION DE VARIABLES

In [5]:
def preseleccion_variables(x, y, n_variables=75):
    #Solo en entrenamiento: se queda con las n variables con mas mutual information contra las ventas
    x = x.drop(columns=['producto'], errors='ignore')
    #un valor de MI por columna, con el nombre de la variable como indice
    mi = pd.Series(mutual_info_regression(x, y, random_state = 42), index=x.columns)
    #las n mas altas
    entran = mi.nlargest(n_variables).index
    return x[entran]

### MODELIZACION

#### ENTRENAMIENTO

In [6]:
def modelizar(x_producto, y):
    '''
    Esta función es la que hace la modelización individual.

    Recibe los datos de las x y la y de un producto.

    Encuentra los parámetros óptimos para ese producto.

    Devuelve el mejor modelo.
    '''
    #Excluye el producto como variable de modelización
    #y preselecciona con MI las variables de ESTE producto (cada serie elige las suyas)
    x_mod = preseleccion_variables(x_producto, y)

    #Define la validación cruzada
    time_cv = TimeSeriesSplit(5, test_size = 8)

    #Define la parrilla de algoritmos
    pipe = Pipeline([('algoritmo',HistGradientBoostingRegressor(random_state = 42))])

    grid = [
         {'algoritmo': [HistGradientBoostingRegressor(random_state = 42)]
#          'algoritmo__learning_rate': [0.01,0.025,0.05,0.1],
#          'algoritmo__max_iter': [50,100,200],
#          'algoritmo__max_depth': [5,10,20,50],
#          'algoritmo__scoring': ['neg_mean_absolute_error'],   #OJO: no descomentar, el scoring va en el buscador, no en el algoritmo
#          'algoritmo__l2_regularization': [0,0.25,0.5,0.75,1]
         }

    ]

    #Crea los modelos
    random_search = RandomizedSearchCV(estimator = pipe,
                                   param_distributions = grid,
                                   n_iter = 1,
                                   cv = time_cv,
                                   scoring = 'neg_mean_absolute_error',
                                   verbose = 0,
                                   n_jobs = -1,
                                   random_state = 42)   #para que el sorteo de combinaciones se pueda repetir

    random_search.fit(x_mod, y)

    #Reentrena el mejor sobre todos los datos
    #(no hace falta otro .fit: RandomizedSearchCV ya lo hace solo con refit=True, que viene por defecto)
    modelo_final = random_search.best_estimator_

    #Devuelve como salida el mejor modelo
    return modelo_final

In [7]:
#Donde se guarda la lista con los 20 modelos entrenados
RUTA_MODELOS = MODELOS / 'lista_modelos_retail.pkl'


def lanzar_entrenamiento(df):

    '''
    Esta función va recorriendo todos los productos y llamando a modelizar() para crear una lista total con todos los modelos de todos los productos.

    Recibe el dataframe que sale de crear_variables(): limpio, con la columna producto y con la target (ventas) dentro.

    No devuelve nada, si no que guarda en disco el objeto ya entrenado con todos los modelos.
    '''

    #Transforma UNA sola vez con todo el historico: el encoder aprende de las 20 series juntas y se guarda una vez
    df = transformar_variables_entrenamiento(df)

    lista_productos = list(df['producto'].unique())

    lista_modelos = []

    for cada in lista_productos:

        #Renombra por claridad
        producto = cada
        target = 'ventas'

        #x e y de este producto, sacadas de sus propias filas (la fecha se repite entre series, isin no sirve)
        x = df.loc[df['producto'] == producto].drop(columns=target).copy()
        y = df.loc[df['producto'] == producto, target].copy()

        #Llama a la funcion de modelizar (dentro ya hace la preseleccion con MI de este producto)
        modelo = modelizar(x, y)

        #Añade el modelo final a la lista
        lista_modelos.append((producto, modelo))

    #Guarda la lista de modelos entrenados
    joblib.dump(lista_modelos, RUTA_MODELOS)

#### EJECUCION

In [8]:
def lanzar_ejecucion(df):
    '''
    Predice todos los productos, pero solo el dia que toca.
    Recibe el dataframe que sale de crear_variables().
    '''
    #carga la lista [(producto, modelo), ...] que guardo lanzar_entrenamiento
    lista_modelos = joblib.load(RUTA_MODELOS)

    #transforma una sola vez con el encoder guardado (igual que en el entrenamiento)
    df = transformar_variables_ejecucion(df)

    lista_predicciones = []

    for cada in lista_modelos:
        #renombra por claridad
        producto = cada[0]
        modelo = cada[1]

        #filas de este producto, solo con las variables que eligio su modelo (y en el mismo orden)
        x = df.loc[df.producto == producto, modelo.feature_names_in_]

        #prediccion de este producto, con la fecha en el indice
        predicciones = pd.DataFrame({'producto': producto,
                                     'prediccion': modelo.predict(x).astype('int')},
                                    index = x.index)

        lista_predicciones.append(predicciones)

    #junta las predicciones de los 20 productos
    predicciones_df = pd.concat(lista_predicciones)

    #el dia que toca es la fecha mas baja que sobrevivio al dropna de los lags
    predicciones_df = predicciones_df.loc[predicciones_df.index == predicciones_df.index.min()]

    #date pasa a columna para el CSV que va a Power BI
    return predicciones_df.reset_index()

In [9]:
#dias a predecir (el horizonte del pedido)
HORIZONTE = 8


def forecasting_autorregresivo(x):
    '''
    No reentrena nada: usa los modelos ya entrenados, como un CRM que pide un scoring a la API.
    Los ultimos 15 dias se aplanan en una sola fila (ventas_lag1 a ventas_lag15) y el modelo da un dia.
    Como mañana no existe, cada prediccion se escribe como venta y se repite hasta completar el HORIZONTE.
    Recibe 15 dias de historia mas los 8 dias a predecir.
    '''
    for cada in range(0, HORIZONTE):
        #prepara la fila de cada producto igual que en el entrenamiento y predice el dia que toca
        paso1_df = calidad_datos(x)
        paso2_df = crear_variables(paso1_df)
        f = lanzar_ejecucion(paso2_df)

        #la prediccion pasa a ser la venta de ese dia, para que el shift la meta en los lags de la siguiente vuelta
        #(cada fila busca la de su producto por nombre, asi no depende del orden)
        prediccion_por_producto = f.set_index('producto')['prediccion']
        dia = f['date'].iloc[0]
        es_dia = x.index == dia
        producto_fila = x.loc[es_dia, 'store_id'] + '_' + x.loc[es_dia, 'item_id']
        x.loc[es_dia, 'ventas'] = producto_fila.map(prediccion_por_producto).values

        #se borra el dia mas antiguo: si no, el mismo dia seguiria con sus 15 lags completos y se predeciria otra vez
        #al borrarlo, la fecha mas baja completa pasa a ser el dia siguiente y la ventana avanza
        x = x.loc[x.index != x.index.min()]

    return x

## PROCESO

### REENTRENAMIENTO

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'


Reentrenamiento despues de importacion

In [11]:
#REENTRENAMIENTO: todo el historico crudo, sin tocar
df = pd.read_pickle(ENTRENAMIENTO / 'train.pkl')

x = calidad_datos(df)
x = crear_variables(x)
lanzar_entrenamiento(x)   #guarda ohe_retail.pkl y lista_modelos_retail.pkl en 05_modelos

### EVALUACION

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'

#Evaluación

#EVALUACION a 8 dias, como en la vida real: nos paramos el 30 de noviembre (el corte)
train = pd.read_pickle(ENTRENAMIENTO / 'train.pkl')
val = pd.read_pickle(VALIDACION / 'validacion.pkl')

#historia contiene los ultimos 15 dias reales de noviembre, los lags se van a llenar hasta el
# 1 de diciembre con el forecasting recursivo
historia = train.loc['2015-11-16':]
#se sacan los 8 dias a predecir de la validación, del 1 al 8 de diciembre
futuro = val.loc[:'2015-12-08'].copy()

#Se rellenan con nulos las ventas en el periodo al 8 de diciembre (la ventana que se definio)
reales = futuro[['store_id', 'item_id', 'ventas']].reset_index()
futuro['ventas'] = np.nan

#se concatena del 16 de nov al 8 de diciembre
entrada = pd.concat([historia, futuro])

# el recursivo predice del 1 al 8, usando exclusivamente los datos del 16 de nov al 30 de noviembre
# aplanandolos y metiendolos desde el 01 /12 al 08 /12 en sus ventaslag15 hacia atras,mientras
# va borrando los valores antiguos de las ventas que alcanzan hasta el 23 de noviembre
# se llenaron los datos del vaciado de las de validacion (futuro) por sencillez
resultado = forecasting_autorregresivo(entrada)

#se localiza solo lo que necesitamos, del 01/12 al 08/12 con
# las columnas necesarias para la comparación con los valores reales (reales)
predichas = resultado.loc['2015-12-01':, ['store_id', 'item_id', 'ventas']].reset_index()
comparar = reales.merge(predichas, on=['date', 'store_id', 'item_id'], suffixes=('_real', '_predicha'))

#Se comapran las dos "Y" (ventas reales vs ventas predichas)
print('MAE a 8 dias = ', mean_absolute_error(comparar['ventas_real'], comparar['ventas_predicha']))
comparar

MAE a 8 dias =  9.55625


,date,store_id,item_id,ventas_real,ventas_predicha
0,2015-12-01,CA_3,FOODS_3_090,57,35.000
1,2015-12-01,CA_3,FOODS_3_120,109,71.000
2,2015-12-01,CA_3,FOODS_3_202,17,16.000
3,2015-12-01,CA_3,FOODS_3_252,29,38.000
4,2015-12-01,CA_3,FOODS_3_288,25,32.000
...,...,...,...,...,...
155,2015-12-08,CA_4,FOODS_3_329,0,5.000
156,2015-12-08,CA_4,FOODS_3_555,8,5.000
157,2015-12-08,CA_4,FOODS_3_586,8,8.000
158,2015-12-08,CA_4,FOODS_3_587,9,14.000


In [23]:
#error de cada fila y su promedio por dia: deberia crecer del 1 al 8 de diciembre
comparar['error'] = (comparar['ventas_real'] - comparar['ventas_predicha']).abs()
comparar.groupby('date')['error'].mean()

date
2015-12-01    8.050
2015-12-02   10.450
2015-12-03   10.450
2015-12-04    9.750
2015-12-05    9.250
2015-12-06   11.950
2015-12-07    7.550
2015-12-08    9.000
Name: error, dtype: float64

### EJECUCION

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'
HORIZONTE = 8


#EJECUCION: la entrada llega lista desde la consulta SQL (CSV en 07_despliegue)
entrada = pd.read_csv(RAIZ / '07_despliegue' / 'DatosParaProduccion.csv', sep=';')

#se pasa la fecha a texto aaaa-mm-dd (igual que en el entrenamiento) y queda como indice
entrada['date'] = pd.to_datetime(entrada['date'], dayfirst=True).dt.strftime('%Y-%m-%d')
entrada = entrada.set_index('date')

# el CSV inicial trae los eventos futuros en 0, se vacian para que calidad_datos los vuelva Sin_evento
entrada[['event_name_1', 'event_type_1']] = entrada[['event_name_1', 'event_type_1']].replace({0: np.nan, '0': np.nan})

#ultimo dia con ventas reales: justo antes de los HORIZONTE dias a predecir,
# hace una especie de DISTINCT a los días presentes en la entrada, donde
# se toma el último día con ventas reales antes del horizonte a predecir
fecha_corte = entrada.index.unique().sort_values()[-(HORIZONTE + 1)]

#el recursivo predice los 8 dias, sobrescribiendo los 0 de ventas con sus predicciones
resultado = forecasting_autorregresivo(entrada)

#solo los dias predichos, con lo que necesita Power BI
#localiza aquellos posteriores a la fecha de corte ya rellenados con predicciones
predicciones = resultado.loc[resultado.index > fecha_corte, ['store_id', 'item_id', 'ventas']].reset_index()
predicciones.to_csv(RAIZ / '07_despliegue' / 'predicciones.csv', index=False)
predicciones   #160 filas: 20 productos x 8 dias

,date,store_id,item_id,ventas
0,2015-12-24,CA_3,FOODS_3_090,8
1,2015-12-24,CA_3,FOODS_3_120,58
2,2015-12-24,CA_3,FOODS_3_202,20
3,2015-12-24,CA_3,FOODS_3_252,38
4,2015-12-24,CA_3,FOODS_3_288,33
...,...,...,...,...
155,2015-12-31,CA_4,FOODS_3_329,6
156,2015-12-31,CA_4,FOODS_3_555,4
157,2015-12-31,CA_4,FOODS_3_586,9
158,2015-12-31,CA_4,FOODS_3_587,10
